# Hiram — Data Pipeline

**Project:** How Much Does Call Duration Leak? Predicting Term-Deposit Subscriptions  
**Course:** CSCI 3345 Machine Learning, Fall 2026  
**Owner:** Hiram Chen

This notebook:
1. Loads UCI Bank Marketing (`bank-additional-full.csv`)
2. Removes duplicate rows
3. Handles `pdays = 999` with a binary indicator
4. Makes a stratified 70 / 15 / 15 split (fixed seed)
5. Fits preprocessing on **train only** (one-hot + standardization)
6. Builds feature matrices **with and without** `duration`
7. Saves the arrays and fitted preprocessors for the modeling notebooks

Run from the repository root or `src/`. The majority baseline is in `02_baseline.ipynb`.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
import joblib

RANDOM_STATE = 42
cwd = Path.cwd()
root = cwd.parent if cwd.name == "src" else cwd
DATA_PATH = root / "data" / "bank-additional-full.csv"
OUT_DIR = root / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Ready.")

## 1. Load data

In [ ]:
df_raw = pd.read_csv(DATA_PATH, sep=";")
print("Raw shape:", df_raw.shape)
print("Columns:", list(df_raw.columns))
print("\nTarget counts:")
print(df_raw["y"].value_counts())
print("\nPositive rate: {:.3%}".format((df_raw["y"] == "yes").mean()))
df_raw.head()

## 2. Clean: drop exact duplicates

In [ ]:
n_dupes = df_raw.duplicated().sum()
print(f"Exact duplicate rows: {n_dupes}")

df = df_raw.drop_duplicates().reset_index(drop=True)
print(f"Shape after dropping duplicates: {df.shape}")
assert n_dupes == 12 or n_dupes >= 1, "Expected ~12 duplicates as in the proposal"
print("Positive rate after drop: {:.3%}".format((df["y"] == "yes").mean()))

## 3. Feature engineering notes

- Keep categorical `"unknown"` as its own category (one-hot later).
- `pdays = 999` means never contacted before → add `pdays_never_contacted` and leave `pdays` as-is for now (or optionally replace 999; we keep both indicator + original for transparency).
- Target: map `yes`→1, `no`→0.

In [ ]:
df = df.copy()
df["pdays_never_contacted"] = (df["pdays"] == 999).astype(int)
print("pdays == 999 rate: {:.3%}".format((df["pdays"] == 999).mean()))
print("New column value counts:")
print(df["pdays_never_contacted"].value_counts())

y = (df["y"] == "yes").astype(int)
print("\ny distribution:")
print(y.value_counts())

## 4. Column groups

In [ ]:
CATEGORICAL = [
    "job", "marital", "education", "default", "housing", "loan",
    "contact", "month", "day_of_week", "poutcome",
]

NUMERIC_BASE = [
    "age", "campaign", "pdays", "previous",
    "emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed",
    "pdays_never_contacted",
]

# duration is leakage — kept only for Q2 ablation
DURATION_COL = ["duration"]

FEATURE_COLS_NO_DURATION = CATEGORICAL + NUMERIC_BASE
FEATURE_COLS_WITH_DURATION = CATEGORICAL + NUMERIC_BASE + DURATION_COL

X_all = df[FEATURE_COLS_WITH_DURATION].copy()

print("Features without duration:", len(FEATURE_COLS_NO_DURATION))
print("Features with duration:", len(FEATURE_COLS_WITH_DURATION))
X_all.head()

## 5. Stratified 70 / 15 / 15 split (fixed seed)

Split once on the full cleaned table, then reuse the saved matrices for model comparisons.

In [ ]:
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_all, y,
    test_size=0.15,
    stratify=y,
    random_state=RANDOM_STATE,
)

# 15% of total ≈ 15/85 of the remaining 85%
val_fraction_of_trainval = 0.15 / 0.85
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=val_fraction_of_trainval,
    stratify=y_trainval,
    random_state=RANDOM_STATE,
)

def summarize_split(name, y_split):
    n = len(y_split)
    pos = y_split.sum()
    print(f"{name:8s}  n={n:5d}  yes={pos:4d}  rate={pos/n:.3%}")

print("Split sizes (target ~70/15/15):")
summarize_split("train", y_train)
summarize_split("val", y_val)
summarize_split("test", y_test)
print(f"\nFractions: train={len(y_train)/len(y):.1%}  val={len(y_val)/len(y):.1%}  test={len(y_test)/len(y):.1%}")

## 6. Preprocessing fitted on **train only**

- Categoricals → one-hot (`handle_unknown="ignore"` so val/test unknowns don't crash)
- Numerics → `StandardScaler`
- Fit two transformers: **no duration** (default / Q1) and **with duration** (Q2)

In [ ]:
def make_preprocessor(numeric_cols):
    return ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL),
            ("num", StandardScaler(), numeric_cols),
        ],
        remainder="drop",
    )

prep_no_dur = make_preprocessor(NUMERIC_BASE)
prep_with_dur = make_preprocessor(NUMERIC_BASE + DURATION_COL)

X_train_no = X_train[FEATURE_COLS_NO_DURATION]
X_val_no = X_val[FEATURE_COLS_NO_DURATION]
X_test_no = X_test[FEATURE_COLS_NO_DURATION]

X_train_with = X_train[FEATURE_COLS_WITH_DURATION]
X_val_with = X_val[FEATURE_COLS_WITH_DURATION]
X_test_with = X_test[FEATURE_COLS_WITH_DURATION]

X_train_no_m = prep_no_dur.fit_transform(X_train_no)
X_val_no_m = prep_no_dur.transform(X_val_no)
X_test_no_m = prep_no_dur.transform(X_test_no)

X_train_with_m = prep_with_dur.fit_transform(X_train_with)
X_val_with_m = prep_with_dur.transform(X_val_with)
X_test_with_m = prep_with_dur.transform(X_test_with)

print("Matrix shapes WITHOUT duration:")
print("  train", X_train_no_m.shape, "val", X_val_no_m.shape, "test", X_test_no_m.shape)
print("Matrix shapes WITH duration:")
print("  train", X_train_with_m.shape, "val", X_val_with_m.shape, "test", X_test_with_m.shape)

## 7. Save artifacts for teammates

Patrick / Braden can load `../data/processed/arrays.npz` and the fitted preprocessors.

In [ ]:
np.savez_compressed(
    OUT_DIR / "arrays.npz",
    X_train_no=X_train_no_m,
    X_val_no=X_val_no_m,
    X_test_no=X_test_no_m,
    X_train_with=X_train_with_m,
    X_val_with=X_val_with_m,
    X_test_with=X_test_with_m,
    y_train=y_train.to_numpy(),
    y_val=y_val.to_numpy(),
    y_test=y_test.to_numpy(),
)

joblib.dump(prep_no_dur, OUT_DIR / "preprocessor_no_duration.joblib")
joblib.dump(prep_with_dur, OUT_DIR / "preprocessor_with_duration.joblib")

# Also keep raw split frames (useful for debugging / time-split later)
X_train.assign(y=y_train.values).to_csv(OUT_DIR / "train_raw.csv", index=False)
X_val.assign(y=y_val.values).to_csv(OUT_DIR / "val_raw.csv", index=False)
X_test.assign(y=y_test.values).to_csv(OUT_DIR / "test_raw.csv", index=False)

meta = {
    "random_state": RANDOM_STATE,
    "n_raw": int(df_raw.shape[0]),
    "n_after_dedupe": int(df.shape[0]),
    "n_duplicates_removed": int(n_dupes),
    "positive_rate": float(y.mean()),
    "split_sizes": {
        "train": int(len(y_train)),
        "val": int(len(y_val)),
        "test": int(len(y_test)),
    },
    "feature_cols_no_duration": FEATURE_COLS_NO_DURATION,
    "feature_cols_with_duration": FEATURE_COLS_WITH_DURATION,
    "primary_metric": "PR-AUC (average_precision)",
    "notes": [
        "Preprocessors fitted on train only.",
        "Default modeling features EXCLUDE duration (leakage).",
        "Use *_with matrices only for Q2 leakage ablation.",
        "Do not use test results for model selection.",
    ],
}
with open(OUT_DIR / "meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Saved to", OUT_DIR.resolve())
print("Files:", sorted(p.name for p in OUT_DIR.iterdir()))

## How teammates load this

```python
import numpy as np, joblib
data = np.load("../data/processed/arrays.npz")
X_train = data["X_train_no"]   # use X_train_with for duration ablation
X_val   = data["X_val_no"]
y_train = data["y_train"]
y_val   = data["y_val"]
# Do not use X_test / y_test for model selection
```

### Hiram write-up bullets (for checkpoint PDF)
- Source: UCI Bank Marketing `bank-additional-full.csv` (41,188 rows).
- Removed exact duplicates (12).
- Added `pdays_never_contacted` for `pdays == 999`.
- Stratified 70/15/15 split, `random_state=42`.
- One-hot + StandardScaler fitted on train only.
- Default features exclude `duration`; with-duration matrices saved for Q2.